# BI IA Dashboard — Pipeline Orquestrador
Este notebook orquestra o fluxo completo:
1) carregar / validar dataset (upload ou Drive / GitHub)
2) pré-processar dados (src.preprocessing)
3) calcular KPIs (src.metrics)
4) visualizar (src.visualization)
5) prever (src.forecasting)
6) gerar relatório automático (src.nlg_agent)
7) exportar resultados (HTML / CSV / JSON)

Execute célula por célula. Para persistência entre sessões, monte o Google Drive.


# 🧰 Instala dependências (Plotly e scikit-learn)
Execute apenas se necessário (Colab normalmente já tem scikit-learn, mas Plotly pode faltar).


In [9]:
!pip install -q plotly scikit-learn pyarrow
print("✅ Dependências: plotly, scikit-learn (verifique instalação se houver erro).")


✅ Dependências: plotly, scikit-learn (verifique instalação se houver erro).


# 🔌 Montar Google Drive (opcional) e/ou clonar repositório
- Se os arquivos `src/` estiverem no seu Drive, monte e aponte BASE_DIR para a pasta do projeto.
- Alternativamente, clone seu repositório GitHub no ambiente /content.


In [10]:
#Célula 3 — Clonar repositório principal e definir BASE_DIR
#Como seu projeto BI IA Dashboard está dentro de:
#vitorhugo-portfolio/Projetos Pessoais/BI IA Dashboard

#Aqui clonamos o repositório raiz e apontamos diretamente para a pasta do projeto.

!git clone https://github.com/vitorsantoszoo/vitorhugo-portfolio.git

import os
import sys

# Ajuste do caminho para a pasta do projeto (com espaços)
BASE_DIR = "/content/vitorhugo-portfolio/Projetos Pessoais/BI IA Dashboard"

# Verifica se o diretório existe
if not os.path.exists(BASE_DIR):
    raise FileNotFoundError(f"❌ O diretório BASE_DIR não foi encontrado:\n{BASE_DIR}")

print("📂 BASE_DIR localizado:", BASE_DIR)

# Adiciona ao sys.path para permitir imports do src/
sys.path.append(BASE_DIR)
sys.path.append(os.path.join(BASE_DIR, "src"))

print("🔧 Path configurado. Tentando importar módulos...")


fatal: destination path 'vitorhugo-portfolio' already exists and is not an empty directory.
📂 BASE_DIR localizado: /content/vitorhugo-portfolio/Projetos Pessoais/BI IA Dashboard
🔧 Path configurado. Tentando importar módulos...


# 📥 Imports e verificação dos módulos src/
Carregamos funções principais dos módulos src/. Se algum import falhar,
verifique se o diretório BASE_DIR/src existe e contém os .py gerados na Etapa B.

In [37]:
try:
    from src.load_data import read_csv_auto, infer_columns, ensure_date
    from src.preprocessing import basic_clean, fill_numeric
    from src.metrics import calc_monthly_kpis
    from src.forecasting import rf_forecast
    from src.visualization import plot_timeseries, plot_top_categories
    from src.nlg_agent import generate_summary

    print("✅ Todos os módulos src importados com sucesso!")

except Exception as e:
    print("❌ ERRO ao importar os módulos:")
    print(e)


✅ Todos os módulos src importados com sucesso!


# 🔁 Seleção do dataset
Opções:
1) Usar o dataset de exemplo em data/examples/sample_sales.csv
2) Fazer upload manual (arquivo CSV)
3) Usar dataset em BASE_DIR/data/user_uploads/

Defina a variável DATA_MODE para 'EXAMPLE', 'UPLOAD' ou 'USER'.


In [12]:
from google.colab import files
DATA_MODE = "EXAMPLE"  # 'EXAMPLE' | 'UPLOAD' | 'USER'
DATA_EXAMPLE_PATH = os.path.join(BASE_DIR, "data", "examples", "sample_sales.csv")
USER_UPLOADS_DIR = os.path.join(BASE_DIR, "data", "user_uploads")

df = None
if DATA_MODE == "EXAMPLE":
    if os.path.exists(DATA_EXAMPLE_PATH):
        df = read_csv_auto(DATA_EXAMPLE_PATH)
        print("Usando dataset de exemplo:", DATA_EXAMPLE_PATH)
    else:
        print("❌ sample_sales.csv não encontrado em:", DATA_EXAMPLE_PATH)
elif DATA_MODE == "UPLOAD":
    print("Abra o diálogo de upload e selecione um arquivo CSV.")
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    df = read_csv_auto(fname)
    # opcional: mover para user_uploads
    dest = os.path.join(USER_UPLOADS_DIR, fname)
    os.makedirs(USER_UPLOADS_DIR, exist_ok=True)
    os.replace(fname, dest)
    print("Arquivo salvo em:", dest)
elif DATA_MODE == "USER":
    # lista arquivos na pasta user_uploads e pega o mais recente (ou pede escolha)
    files_in = glob.glob(os.path.join(USER_UPLOADS_DIR, "*.csv"))
    if not files_in:
        raise FileNotFoundError("Pasta user_uploads está vazia. Faça upload ou coloque um CSV lá.")
    # pega o primeiro (ou pode implementar escolha)
    df = read_csv_auto(files_in[0])
    print("Usando arquivo:", files_in[0])

print("Visualizando as 5 primeiras linhas do dataset:")
display(df.head())
print("\nColunas detectadas:", df.columns.tolist())


Usando dataset de exemplo: /content/vitorhugo-portfolio/Projetos Pessoais/BI IA Dashboard/data/examples/sample_sales.csv
Visualizando as 5 primeiras linhas do dataset:


,date,sales,qty,product,category,region,profit
0,2023-01-01,19034.00,4,Mouse Wireless,Acessórios,Sudeste,5386.59
1,2023-01-01,3024.57,1,Webcam HD,Eletrônicos,Sudeste,639.57
2,2023-01-01,1153.83,3,Webcam HD,Eletrônicos,Sudeste,365.24
3,2023-01-01,3200.19,4,Smart Monitor 27,Eletrônicos,Sul,840.76
4,2023-01-01,4852.56,1,Webcam HD,Eletrônicos,Sul,1495.12



Colunas detectadas: ['date', 'sales', 'qty', 'product', 'category', 'region', 'profit']


# 🔎 Inferência de colunas
Tentamos detectar data, colunas numéricas e categóricas automaticamente.
Se a inferência não encontrar as colunas corretas, o usuário pode definir manualmente.


In [14]:
inferred = infer_columns(df)
print("Colunas inferidas:")
print(inferred)

# estratégia simples: se existir coluna 'date' ou 'data' usamos ela; valor usamos 'sales' ou a primeira numérica
date_col = None
value_col = None

# tenta escolher automaticamente
if inferred["date_cols"]:
    date_col = inferred["date_cols"][0]
else:
    # tenta encontrar padrão
    for c in df.columns:
        if 'date' in c.lower() or 'data' in c.lower():
            date_col = c
            break

numeric_candidates = inferred["numeric_cols"]
if 'sales' in df.columns:
    value_col = 'sales'
elif 'faturamento' in df.columns:
    value_col = 'faturamento'
elif numeric_candidates:
    value_col = numeric_candidates[0]

print(f"\nColuna escolhida automaticamente para data: {date_col}")
print(f"Coluna escolhida automaticamente para valor: {value_col}")

# Caso queira alterar manualmente, defina aqui:
# date_col = "date"
# value_col = "sales"


Colunas inferidas:
{'date_cols': ['date'], 'numeric_cols': ['sales', 'qty', 'profit'], 'categorical_cols': ['product', 'category', 'region']}

Coluna escolhida automaticamente para data: date
Coluna escolhida automaticamente para valor: sales


# 🧹 Pré-processamento
- limpeza básica
- conversão da coluna de data
- preenchimento de numéricos


In [16]:
import pandas as pd

# limpeza básica
df_clean = basic_clean(df)
print("Após basic_clean — shape:", df_clean.shape)

# converter data
if date_col is None:
    raise ValueError("Coluna de data não definida. Defina date_col manualmente.")
df_clean = ensure_date(df_clean, date_col)

# preencher numéricos
df_clean = fill_numeric(df_clean, strategy='median')

# remover linhas sem data ou sem valor
df_clean = df_clean.dropna(subset=[date_col, value_col])
df_clean[date_col] = pd.to_datetime(df_clean[date_col])

print("Preview após limpeza:")
display(df_clean.head())


Após basic_clean — shape: (5065, 7)
Preview após limpeza:


/content/vitorhugo-portfolio/Projetos Pessoais/BI IA Dashboard/src/preprocessing.py:32: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[c].fillna(df[c].median(), inplace=True)


,date,sales,qty,product,category,region,profit
0,2023-01-01,19034.00,4,Mouse Wireless,Acessórios,Sudeste,5386.59
1,2023-01-01,3024.57,1,Webcam HD,Eletrônicos,Sudeste,639.57
2,2023-01-01,1153.83,3,Webcam HD,Eletrônicos,Sudeste,365.24
3,2023-01-01,3200.19,4,Smart Monitor 27,Eletrônicos,Sul,840.76
4,2023-01-01,4852.56,1,Webcam HD,Eletrônicos,Sul,1495.12


# 🧾 KPIs e agregações
Calcula KPIs mensais usando src.metrics.calc_monthly_kpis


In [18]:
monthly, kpis = calc_monthly_kpis(df_clean, date_col=date_col, value_col=value_col)
print("KPIs calculados:")
print(kpis)

# mostra tabela agregada
display(monthly.tail(12))


KPIs calculados:
{'total_periodo': 32125157.48, 'media_mensal': 1784730.971111111, 'crescimento_ultimo_mes': -0.1260729434770187}


/content/vitorhugo-portfolio/Projetos Pessoais/BI IA Dashboard/src/metrics.py:17: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly = s.resample("M").sum()


,sales,pct_change
date,,
2023-07-31,1590193.59,-0.007505
2023-08-31,2017223.13,0.268539
2023-09-30,1698646.42,-0.157928
2023-10-31,1841766.32,0.084255
2023-11-30,1755592.88,-0.046788
2023-12-31,1954340.32,0.113208
2024-01-31,1949207.78,-0.002626
2024-02-29,1811835.41,-0.070476
2024-03-31,1896288.72,0.046612


# 📈 Visualizações Interativas — BI IA Dashboard

Nesta seção geramos os principais gráficos de análise exploratória:

1. **Faturamento Mensal (série temporal)**
   - Mostra a evolução do faturamento ao longo do tempo.
   - Ajuda a identificar tendências, sazonalidade e quedas.

2. **Top Produtos por Faturamento**
   - Lista os produtos que mais geram receita.
   - Permite identificar “campeões de venda”.

3. **Top Categorias por Faturamento**
   - Agrupa por categoria e soma o faturamento.
   - Útil para análise macro do portfólio.

Cada gráfico aparece separado com uma linha divisória abaixo.



In [20]:
# ============ 1) Faturamento Mensal ============
print("\n\n📌 1) Faturamento Mensal — Série Temporal")
print("Mostra a evolução do faturamento mês a mês, indicando tendências e sazonalidade.\n")

monthly_df = monthly.reset_index()

try:
    fig_ts = plot_timeseries(
        monthly_df,
        x=date_col,
        y=value_col,
        title="📈 Faturamento Mensal"
    )
    fig_ts.show()
except Exception as e:
    print("Erro ao plotar time series:", e)

print("\n" + "-"*120 + "\n")


# ============ 2) Top Produtos ============
if 'product' in df_clean.columns:
    print("📌 2) Top Produtos — Ranking por Faturamento")
    print("Mostra os produtos que mais faturaram no período analisado.\n")

    try:
        fig_prod = plot_top_categories(
            df_clean,
            cat_col='product',
            value_col=value_col,
            top_n=10
        )
        fig_prod.update_layout(title="🏆 Top 10 Produtos por Faturamento")
        fig_prod.show()
    except Exception as e:
        print("Erro ao plotar produtos:", e)

    print("\n" + "-"*120 + "\n")


# ============ 3) Top Categorias ============
if 'category' in df_clean.columns:
    print("📌 3) Top Categorias — Faturamento por Categoria")
    print("Agrupa e compara categorias para identificar as mais relevantes.\n")

    try:
        fig_cat = plot_top_categories(
            df_clean,
            cat_col='category',
            value_col=value_col,
            top_n=8
        )
        fig_cat.update_layout(title="📦 Top Categorias por Faturamento")
        fig_cat.show()
    except Exception as e:
        print("Erro ao plotar categorias:", e)

    print("\n" + "-"*120 + "\n")




📌 1) Faturamento Mensal — Série Temporal
Mostra a evolução do faturamento mês a mês, indicando tendências e sazonalidade.




------------------------------------------------------------------------------------------------------------------------

📌 2) Top Produtos — Ranking por Faturamento
Mostra os produtos que mais faturaram no período analisado.




------------------------------------------------------------------------------------------------------------------------

📌 3) Top Categorias — Faturamento por Categoria
Agrupa e compara categorias para identificar as mais relevantes.




------------------------------------------------------------------------------------------------------------------------



# 🔮 Previsão (Forecast)
Cria previsão para os próximos N períodos (meses) usando rf_forecast do src.forecasting.
Usa a série agregada mensal para treinar.


In [46]:
import importlib, src.forecasting
importlib.reload(src.forecasting)

from src.forecasting import rf_forecast_with_ci

print("🎉 Função rf_forecast_with_ci importada com sucesso!")


🎉 Função rf_forecast_with_ci importada com sucesso!


In [45]:
# 🔮 PREVISÃO (FORECAST) COM INTERVALO DE CONFIANÇA E VISUAL PREMIUM

import pandas as pd
import plotly.graph_objects as go

# ---------------- PRINTS EXPLICATIVOS ----------------
print("\n📌 PREVISÃO DE FATURAMENTO (COM INTERVALO DE CONFIANÇA)\n")
print("Este gráfico mostra:\n")
print("• A evolução histórica do faturamento (linha azul)")
print("• A previsão dos próximos meses usando RandomForest (linha laranja)")
print("• Um intervalo de confiança de 90% calculado via bootstrap (faixa sombreada)")
print("\nEste formato é usado em dashboards corporativos e BI para análises estratégicas.\n")


# ---------------- 1) Preparar série ----------------
series = monthly[value_col].copy()
series.name = value_col

# ---------------- 2) Prever (com intervalo) ----------------
n_periods = 6
pred_mean, pred_low, pred_up = rf_forecast_with_ci(
    series,
    n_periods=n_periods,
    n_bootstrap=80
)

# ---------------- 3) Criar datas futuras ----------------
future_dates = pd.date_range(
    start=monthly.index[-1] + pd.offsets.MonthBegin(1),
    periods=n_periods,
    freq="MS"
)

forecast_df = pd.DataFrame({
    date_col: future_dates,
    "forecast": pred_mean,
    "lower": pred_low,
    "upper": pred_up
})

print("🔎 Previsões geradas:")
display(forecast_df)


# ---------------- 4) Criar DataFrame combinado ----------------
history_df = monthly.reset_index()[[date_col, value_col]]
history_df["type"] = "Histórico"

forecast_plot_df = forecast_df.rename(columns={"forecast": value_col})
forecast_plot_df["type"] = "Previsão"

combined_df = pd.concat([history_df, forecast_plot_df], ignore_index=True)


# ---------------- 5) Gráfico premium (estilo BI) ----------------
fig = go.Figure()

# Série histórica
fig.add_trace(go.Scatter(
    x=history_df[date_col],
    y=history_df[value_col],
    mode="lines+markers",
    name="Histórico",
    line=dict(color="#1f77b4", width=3)
))

# Intervalo de confiança (faixa sombreada)
fig.add_trace(go.Scatter(
    x=list(future_dates) + list(future_dates[::-1]),
    y=list(forecast_df["upper"]) + list(forecast_df["lower"][::-1]),
    fill="toself",
    fillcolor="rgba(255,165,0,0.25)",
    line=dict(color="rgba(255,165,0,0)"),
    hoverinfo="skip",
    name="Intervalo de Confiança (90%)"
))

# Linha de previsão
fig.add_trace(go.Scatter(
    x=future_dates,
    y=pred_mean,
    mode="lines+markers",
    name="Previsão",
    line=dict(color="orange", width=3)
))

fig.update_layout(
    title="🔮 Previsão de Faturamento — Histórico + Próximos Meses (com IC 90%)",
    xaxis_title="Data",
    yaxis_title="Faturamento",
    template="plotly_white",
    legend=dict(title="Legenda"),
    height=600
)

fig.show()

print("\n" + "-"*120 + "\n")



📌 PREVISÃO DE FATURAMENTO (COM INTERVALO DE CONFIANÇA)

Este gráfico mostra:

• A evolução histórica do faturamento (linha azul)
• A previsão dos próximos meses usando RandomForest (linha laranja)
• Um intervalo de confiança de 90% calculado via bootstrap (faixa sombreada)

Este formato é usado em dashboards corporativos e BI para análises estratégicas.

🔎 Previsões geradas:


,date,forecast,lower,upper
0,2024-07-01,1.622394e+06,1.606823e+06,1.640801e+06
1,2024-08-01,1.921707e+06,1.907741e+06,1.929135e+06
2,2024-09-01,1.714686e+06,1.700412e+06,1.722657e+06
3,2024-10-01,1.857328e+06,1.840426e+06,1.864555e+06
4,2024-11-01,1.760022e+06,1.745487e+06,1.769699e+06
5,2024-12-01,1.698324e+06,1.682184e+06,1.717993e+06



------------------------------------------------------------------------------------------------------------------------



# 🧾 Relatório automático (NLG)
Gera resumo textual usando src.nlg_agent.generate_summary


In [47]:
# prepara kpis resumidos para o nlg
kpis_for_nlg = {
    "total_periodo": kpis.get("total_periodo", 0),
    "media_mensal": kpis.get("media_mensal", 0),
    "crescimento_ultimo_mes": kpis.get("crescimento_ultimo_mes", 0)
}
report_text = generate_summary(kpis_for_nlg, preds)
print(report_text)


📊 Relatório Inteligente — BI IA Dashboard
📅 Data: 2025-11-13

💰 Faturamento total analisado: R$ 32,125,157.48
📅 Média mensal: R$ 1,784,730.97
📈 Variação do último mês: -12.61% (queda)

🔮 Previsão (próximos períodos):
[np.float64(1625567.43), np.float64(1917839.03), np.float64(1713313.36), np.float64(1839981.45), np.float64(1753293.66), np.float64(1725569.41)]

💡 Insight:
- Caso a tendência continue em queda, recomenda-se revisão de estoque e campanhas.
- Focar nos produtos mais rentáveis e regiões de maior desempenho.


# 💾 Exportar resultados
Salva:
- forecast_df -> CSV
- report_text -> TXT
- monthly -> CSV
Arquivos são salvos em BASE_DIR/dashboard/exports e em data/logs (opcional).


In [48]:
exports_dir = os.path.join(BASE_DIR, "dashboard", "exports")
os.makedirs(exports_dir, exist_ok=True)

forecast_csv = os.path.join(exports_dir, "forecast.csv")
monthly_csv = os.path.join(exports_dir, "monthly_agg.csv")
report_txt = os.path.join(exports_dir, f"report_{pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')}.txt")

forecast_df.to_csv(forecast_csv)
monthly.to_csv(monthly_csv)
with open(report_txt, "w", encoding="utf-8") as f:
    f.write(report_text)

print("📁 Arquivos exportados em:", exports_dir)
print(" -", forecast_csv)
print(" -", monthly_csv)
print(" -", report_txt)


📁 Arquivos exportados em: /content/vitorhugo-portfolio/Projetos Pessoais/BI IA Dashboard/dashboard/exports
 - /content/vitorhugo-portfolio/Projetos Pessoais/BI IA Dashboard/dashboard/exports/forecast.csv
 - /content/vitorhugo-portfolio/Projetos Pessoais/BI IA Dashboard/dashboard/exports/monthly_agg.csv
 - /content/vitorhugo-portfolio/Projetos Pessoais/BI IA Dashboard/dashboard/exports/report_20251113_220316.txt


# 🔁 Exportar um HTML simples com gráficos Plotly
Concatena algumas figuras em uma página HTML básica.


In [49]:
from plotly.offline import plot

html_path = os.path.join(exports_dir, "dashboard_preview.html")
with open(html_path, "w", encoding="utf-8") as f:
    f.write("<html><head><meta charset='utf-8'><title>BI IA Dashboard</title></head><body>\n")
    # timeseries
    try:
        inner_html = plot(fig_ts, include_plotlyjs='cdn', output_type='div')
        f.write("<h2>Faturamento Mensal</h2>\n")
        f.write(inner_html)
    except:
        pass
    # products
    try:
        inner_html = plot(fig_prod, include_plotlyjs=False, output_type='div')
        f.write("<h2>Top Produtos</h2>\n")
        f.write(inner_html)
    except:
        pass
    # categories
    try:
        inner_html = plot(fig_cat, include_plotlyjs=False, output_type='div')
        f.write("<h2>Top Categorias</h2>\n")
        f.write(inner_html)
    except:
        pass

    f.write("</body></html>")

print("✅ Dashboard HTML salvo em:", html_path)


✅ Dashboard HTML salvo em: /content/vitorhugo-portfolio/Projetos Pessoais/BI IA Dashboard/dashboard/exports/dashboard_preview.html


# ✅ Pipeline finalizado
O notebook gerou:
- KPIs mensais
- Gráficos interativos
- Previsão para os próximos N períodos
- Relatório automático (texto)
- Arquivos exportados em dashboard/exports

Próximos passos (opcionais):
- integrar modelo Prophet para séries temporais mais robustas
- adicionar widget interativo (ipywidgets) para seleção de colunas no notebook
- adicionar opção de exportar PDF (weasyprint / wkhtmltopdf / nbconvert)


In [50]:
print("🎯 Pipeline executado. Verifique a pasta de exports e os gráficos gerados.")
print("Se quiser, eu deixo este notebook com widgets interativos (seletor de colunas, botões) — quer que eu adicione?")


🎯 Pipeline executado. Verifique a pasta de exports e os gráficos gerados.
Se quiser, eu deixo este notebook com widgets interativos (seletor de colunas, botões) — quer que eu adicione?
